In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

#### Load Cleaned Dataset
import pandas as pd

data = pd.read_parquet(
    "/content/drive/My Drive/NSW_Electricity_Demand/Data/Processed/fact_merged_clean.parquet"
)

In [5]:
import numpy as np
import pandas as pd



#### Sort Data by Datetime
data['DATETIME'] = pd.to_datetime(data['DATETIME'])
data = data.sort_values('DATETIME').reset_index(drop=True)

#### Hour Month Year
data['HOUR']  = data['DATETIME'].dt.hour
data["MONTH"] = data["DATETIME"].dt.month
data["DAY"]   = data["DATETIME"].dt.day


#### Temprature
data["TEMP2"] = data["TEMPERATURE"] ** 2
data["TEMP3"] = data["TEMPERATURE"] ** 3




# Rolling demand statistics (based on 30-min steps)
data["ROLL_MEAN_12"] = data["TOTALDEMAND"].rolling(12).mean()    # 6 hours
data["ROLL_MEAN_24"] = data["TOTALDEMAND"].rolling(24).mean()    # 12 hours

# --- Rolling Standard Deviations ---
# 12 intervals = 6 hours (since 30-min intervals)
# 24 intervals = 12 hours
data["ROLL_STD_12"] = data["TOTALDEMAND"].rolling(window=12, min_periods=1).std()
data["ROLL_STD_24"] = data["TOTALDEMAND"].rolling(window=24, min_periods=1).std()



#### TOTALDEMAND Lag Features
for i in range(1, 49):  # 1 to 48 lags (each 30 min apart)
    data[f'Lag_{i}'] = data['TOTALDEMAND'].shift(i)

#### Weekend

data["IS_WEEKEND"] = data["DATETIME"].dt.weekday >= 5
data["IS_WEEKEND"] = data["IS_WEEKEND"].astype(int)

#### Peak

hourly_avg = data.groupby("HOUR")["TOTALDEMAND"].mean()
threshold = hourly_avg.quantile(0.75)
peak_hours = hourly_avg[hourly_avg >= threshold].index.tolist()
print("Auto-detected peak hours:", peak_hours)
data["IS_PEAK"] = data["DATETIME"].dt.hour.isin(peak_hours).astype(int)

#### Define and subset the columns you want to keep


keep_cols = ['PREDISPATCHSEQNO', 'REGIONID', 'PERIODID', 'FORECASTDEMAND', 'LASTCHANGED', 'DATETIME', 'TOTALDEMAND','LOCATION',
             'TEMPERATURE', "TEMP2", "TEMP3",
             'ROLL_MEAN_12', 'ROLL_MEAN_24', 'ROLL_STD_12','ROLL_STD_24','IS_WEEKEND', 'HOUR', 'IS_PEAK', 'DAY', 'MONTH',
             "Lag_1","Lag_2","Lag_3","Lag_4","Lag_5","Lag_6","Lag_7","Lag_8","Lag_9","Lag_10","Lag_11","Lag_12","Lag_13",
            "Lag_14","Lag_15","Lag_16","Lag_17","Lag_18","Lag_19","Lag_20","Lag_21","Lag_22","Lag_23","Lag_24","Lag_25",
            "Lag_26","Lag_27","Lag_28","Lag_29","Lag_30","Lag_31","Lag_32","Lag_33","Lag_34","Lag_35","Lag_36","Lag_37",
            "Lag_38","Lag_39","Lag_40","Lag_41","Lag_42","Lag_43","Lag_44","Lag_45","Lag_46","Lag_47","Lag_48"
             ]


# Keep only those columns that actually exist in your dataset
data = data[[c for c in keep_cols if c in data.columns]].copy()


#### Drop rows with any missing (NaN) values
before = data.shape[0]
data = data.dropna().reset_index(drop=True)
after = data.shape[0]


print(f"Dropped {before - after} rows with NaNs.")
print(f"Final shape: {data.shape}")
print(f"Columns ({len(data.columns)}): {list(data.columns)}")

#### Save the processed dataset

data.to_parquet("/content/drive/My Drive/NSW_Electricity_Demand/Data/Processed/Data.parquet", index=False)

Auto-detected peak hours: [9, 16, 17, 18, 19, 20]
Dropped 48 rows with NaNs.
Final shape: (196417, 68)
Columns (68): ['PREDISPATCHSEQNO', 'REGIONID', 'PERIODID', 'FORECASTDEMAND', 'LASTCHANGED', 'DATETIME', 'TOTALDEMAND', 'LOCATION', 'TEMPERATURE', 'TEMP2', 'TEMP3', 'ROLL_MEAN_12', 'ROLL_MEAN_24', 'ROLL_STD_12', 'ROLL_STD_24', 'IS_WEEKEND', 'HOUR', 'IS_PEAK', 'DAY', 'MONTH', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Lag_5', 'Lag_6', 'Lag_7', 'Lag_8', 'Lag_9', 'Lag_10', 'Lag_11', 'Lag_12', 'Lag_13', 'Lag_14', 'Lag_15', 'Lag_16', 'Lag_17', 'Lag_18', 'Lag_19', 'Lag_20', 'Lag_21', 'Lag_22', 'Lag_23', 'Lag_24', 'Lag_25', 'Lag_26', 'Lag_27', 'Lag_28', 'Lag_29', 'Lag_30', 'Lag_31', 'Lag_32', 'Lag_33', 'Lag_34', 'Lag_35', 'Lag_36', 'Lag_37', 'Lag_38', 'Lag_39', 'Lag_40', 'Lag_41', 'Lag_42', 'Lag_43', 'Lag_44', 'Lag_45', 'Lag_46', 'Lag_47', 'Lag_48']
